# Propagation-graph detectors with richer features (GPU)

Trains the graph detectors (GCN / GAT / **Bi-GCN**) on UPFD with the **`bert` node features** (768-dim text embeddings per account, instead of the 10-dim `profile` features used in the CPU baselines), on both UPFD datasets: **politifact** and the larger **gossipcop**.

Same architectures and training protocol as `src/factnet/graph/` (CPU reference on politifact + `profile`: GCN 0.765, GAT 0.765, Bi-GCN **0.774** macro-F1).

**How to run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. About 10–15 min (most of it dataset download). The last cell downloads `graph_metrics.json`.

In [ ]:
%pip install -q torch-geometric

import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Models, identical to src/factnet/graph/models.py
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool


class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden, num_classes):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin = Linear(hidden, num_classes)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)


class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden, num_classes, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=heads)
        self.conv2 = GATConv(hidden * heads, hidden, heads=1)
        self.lin = Linear(hidden, num_classes)

    def forward(self, x, edge_index, batch):
        x = F.elu(self.conv1(x, edge_index))
        x = F.elu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)


class BiGCN(torch.nn.Module):
    """Bidirectional GCN: top-down on the propagation edges, bottom-up on the
    reversed edges, pooled and concatenated (Bian et al., 2020)."""

    def __init__(self, in_dim, hidden, num_classes):
        super().__init__()
        self.td1 = GCNConv(in_dim, hidden)
        self.td2 = GCNConv(hidden, hidden)
        self.bu1 = GCNConv(in_dim, hidden)
        self.bu2 = GCNConv(hidden, hidden)
        self.lin = Linear(hidden * 2, num_classes)

    def forward(self, x, edge_index, batch):
        rev = edge_index.flip(0)
        td = F.relu(self.td1(x, edge_index))
        td = F.relu(self.td2(td, edge_index))
        bu = F.relu(self.bu1(x, rev))
        bu = F.relu(self.bu2(bu, rev))
        td = global_mean_pool(td, batch)
        bu = global_mean_pool(bu, batch)
        return self.lin(torch.cat([td, bu], dim=1))


MODELS = {"GCN": GCN, "GAT": GAT, "Bi-GCN": BiGCN}

In [ ]:
# Training protocol, identical to src/factnet/graph/train_upfd.py (device-aware)
from sklearn.metrics import accuracy_score, f1_score
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader


def run(model_cls, train_ds, test_ds, epochs=60, hidden=64, lr=0.01, seed=0):
    torch.manual_seed(seed)
    model = model_cls(train_ds.num_features, hidden, train_ds.num_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=128)

    for _ in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            F.cross_entropy(out, batch.y).backward()
            opt.step()

    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            pred = model(batch.x, batch.edge_index, batch.batch).argmax(dim=1)
            ps += pred.cpu().tolist()
            ys += batch.y.cpu().tolist()
    return accuracy_score(ys, ps), f1_score(ys, ps, average="macro")

In [ ]:
SETUPS = [
    ("politifact", "profile"),  # CPU-baseline setting, sanity check on GPU
    ("politifact", "bert"),
    ("gossipcop", "profile"),
    ("gossipcop", "bert"),
]

results = []
for name, feature in SETUPS:
    train_ds = UPFD("/tmp/upfd", name, feature, split="train")
    test_ds = UPFD("/tmp/upfd", name, feature, split="test")
    print(f"\n=== {name} / {feature}  (train {len(train_ds)}, test {len(test_ds)}, "
          f"feat {train_ds.num_features}) ===")
    for model_name, cls in MODELS.items():
        acc, f1 = run(cls, train_ds, test_ds)
        tag = "  (main)" if model_name == "Bi-GCN" else ""
        print(f"  {model_name:10s} acc {acc:.3f}  macro-F1 {f1:.3f}{tag}")
        results.append({"dataset": name, "feature": feature, "model": model_name,
                        "accuracy": round(acc, 3), "macro_f1": round(f1, 3)})

In [ ]:
import json

print(f"{'dataset':12s} {'feature':9s} {'model':8s} {'accuracy':>9s} {'macro-F1':>9s}")
print("-" * 52)
for r in results:
    print(f"{r['dataset']:12s} {r['feature']:9s} {r['model']:8s} "
          f"{r['accuracy']:>9.3f} {r['macro_f1']:>9.3f}")

with open("graph_metrics.json", "w") as f:
    json.dump({"benchmark": "UPFD", "runs": results}, f, indent=2)

try:
    from google.colab import files
    files.download("graph_metrics.json")
except ImportError:
    print("saved graph_metrics.json")